# Issue #68 Phase 2 — データ準備 (So-Fake-Set 層化 subset + license監査)

学習パイロット用の学習データを Drive `DETECTER-training/` に構築する。

- **データ**: So-Fake-Set (HF `hzlsaber/So-Fake-Set`) の REAL / FULL_SYNTHETIC 層化 subset (REAL 15K + SYN 15K = 30K)、TAMPERED は除外
- **license-clean**: upstream `source` が CelebA / FFHQ / OpenForensics / WIDER (non-commercial系) の行を除外して `train_clean.jsonl` を作る。除外は画像ではなくメタデータのsource名で判定
- **degraded view**: 学習時に on-the-fly で生成 (本notebookではtransform定義のみ、画像は作らない)
- **OOD保護**: So-Fake-OOD には一切触れない (GPT-Image-2/Nano Banana は OOD 側)。自前 locked コホート 715群も学習・閾値調整に使わない

出力: `MyDrive/DETECTER-training/{train_clean.jsonl,train_subset.tar,license_report.json,degraded_transform.py,prep_report.json}`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, tarfile, hashlib, collections

BASE = '/content/drive/MyDrive/DETECTER-training'
os.makedirs(BASE, exist_ok=True)
SEED = 20260927
random.seed(SEED)

# tuning 目標 (層化 subset 規模)
N_REAL = 15000
N_SYN  = 15000
SYN_PER_GEN = 500          # full-synthetic 生成器あたりの上限
# license 上の非商用/再配布制限が強い upstream real source (Issue #68 調査由来)
RESTRICTED_SOURCES = {'celeba', 'ffhq', 'openforensics', 'wider', 'widerface'}
print('BASE =', BASE)

## 1. So-Fake-Set メタデータ読み込み + license監査

HF viewer API (`/rows`) で全行メタデータを集め、source 別件数を出す。
行全体は `image.bytes` などを含むため、画像は含まない軽い列だけを問い合わせる。
API 上限 100行/リクエストなので offset を回す (全量 ~2.2M 行あるためここでは config/split の列名を確認して parquet 直読みに切り替える)。

In [ ]:
import requests, json
DATASET = 'hzlsaber/So-Fake-Set'
# 1) config/split 一覧
r = requests.get(f'https://datasets-server.huggingface.co/splits?dataset={DATASET}', timeout=60)
r.raise_for_status()
splits = r.json()
for s in splits['splits']:
    print(s['config'], s['split'], s.get('num_examples'))


In [ ]:
# 2) parquet ファイル一覧 -> train のメタのみ読む
r = requests.get(f'https://huggingface.co/api/datasets/{DATASET}/tree/main', timeout=60)
r.raise_for_status()
files = [x['path'] for x in r.json() if x['type']=='file']
print('\n'.join(files[:40]))


In [ ]:
!pip install -q pyarrow pandas huggingface_hub
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq
import pandas as pd

# parquet が見つかった場合: 先頭ファイルをDLして列名を確認
# (行数が多いので metadata-only の列だけ read_table(..., columns=[...]) で読む)
parquets = [f for f in files if f.endswith('.parquet')]
print('parquet files:', len(parquets), parquets[:8])

In [ ]:
# parquet のスキーマだけ確認 (画像列を除く列名を特定)
p0 = hf_hub_download(DATASET, parquets[0], repo_type='dataset')
schema = pq.read_schema(p0)
print(schema)

In [ ]:
# 全 parquet を列限定で読み、REAL vs FULL_SYNTHETIC に絞って層化抽出
# 列名はスキーマ確認結果に合わせて調整する
META_COLS = None  # 例: ['image_id','label','generator','source','split','group_id']
rows = []
for f in parquets:
    p = hf_hub_download(DATASET, f, repo_type='dataset')
    cols = META_COLS or [c for c in schema.names if c != 'image']
    t = pq.read_table(p, columns=cols)
    rows.append(t.to_pandas())
meta = pd.concat(rows, ignore_index=True)
print(len(meta), meta.columns.tolist())
print(meta.head(3))

In [ ]:
# --- license監査: real行の upstream source 分布
# 列名は実スキーマに合わせる (例: label/category/source/generator)
print(meta['label'].value_counts().head())
for col in ('source','category','generator'):
    if col in meta.columns:
        print('\n==', col)
        print(meta.loc[meta['label'].str.contains('real', case=False, na=False), col].value_counts().head(20))

In [ ]:
# --- license-clean subset の確定
def is_restricted(src):
    if not isinstance(src, str): return False
    s = src.lower()
    return any(k in s for k in RESTRICTED_SOURCES)

real = meta[meta['label'].str.contains('real', case=False, na=False)].copy()
syn  = meta[meta['label'].str.contains('full', case=False, na=False)
      | meta['label'].str.contains('synthetic', case=False, na=False)].copy()

real['restricted'] = real['source'].map(is_restricted) if 'source' in real.columns else False
report = {
    'total_rows': len(meta),
    'real_rows': len(real),
    'real_restricted': int(real['restricted'].sum()),
    'real_clean': int((~real['restricted']).sum()),
    'synthetic_rows': len(syn),
    'restricted_sources': sorted(RESTRICTED_SOURCES),
    'per_source_counts': real['source'].value_counts().to_dict() if 'source' in real.columns else {},
}
print(json.dumps(report, ensure_ascii=False, indent=2)[:2000])
with open(f'{BASE}/license_report.json','w') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

clean_real = real[~real['restricted']]
print('clean real:', len(clean_real), 'syn:', len(syn))

In [ ]:
# --- 層化抽出: real N_REAL、syn は generator あたり SYN_PER_GEN 上限で N_SYN
sel_real = clean_real.sample(n=min(N_REAL, len(clean_real)), random_state=SEED)

syn_per = (syn.groupby('generator', group_keys=False)
              .apply(lambda g: g.sample(n=min(SYN_PER_GEN, len(g)), random_state=SEED))
             if 'generator' in syn.columns else syn.sample(n=min(N_SYN, len(syn)), random_state=SEED))
sel_syn = syn_per.sample(n=min(N_SYN, len(syn_per)), random_state=SEED)

subset = pd.concat([sel_real, sel_syn]).reset_index(drop=True)
subset['split_role'] = 'train'
subset.to_json(f'{BASE}/train_clean.jsonl', orient='records', lines=True)
print('subset:', len(subset), 'real:', len(sel_real), 'syn:', len(sel_syn),
      'generators:', sel_syn['generator'].nunique() if 'generator' in sel_syn.columns else 'n/a')

## 2. degraded view transform (on-the-fly 学習時適用)

`build` 後の学習スクリプトで `paired CE` に使う劣化を定義。評価用 1024/Q75 コホートと SNS multi-hop holdout には使わない。評価側の加工は別系統。

In [ ]:
TRANSFORM_SRC = r'''
# degraded_transform.py — paired-CE/KL 用の on-the-fly 劣化
# clean画像 Tensor/PIL -> degraded PIL。評価コホート (1024/Q75, SNS multi-hop) とは別系統。
import io, random
from PIL import Image, ImageEnhance

def degraded_view(img: Image.Image, rng: random.Random) -> Image.Image:
    # 長辺縮小 (768〜1280)、JPEG Q (60〜90)、WebP化確率0.3、色シフト、軽いcanvas/スクショ風border
    if rng.random() < 0.9:
        img = img.copy()
        img.thumbnail((rng.randint(768,1280),)*2, Image.LANCZOS)
    # color shift
    if rng.random() < 0.4:
        img = ImageEnhance.Color(img).enhance(0.7 + rng.random()*0.6)
        img = ImageEnhance.Contrast(img).enhance(0.8 + rng.random()*0.4)
    # re-encode: JPEG Q75中心 ±15 / 稀にWebP
    buf = io.BytesIO()
    if rng.random() < 0.3:
        img.save(buf, 'WEBP', quality=rng.randint(60,90))
    else:
        q = max(40, min(95, int(rng.gauss(75, 10))))
        img.convert('RGB').save(buf, 'JPEG', quality=q)
    buf.seek(0)
    return Image.open(buf).convert('RGB')
'''
with open(f'{BASE}/degraded_transform.py','w') as f:
    f.write(TRANSFORM_SRC)
print('wrote degraded_transform.py')

In [ ]:
# --- prep_report.json
prep = {
    'dataset': DATASET, 'seed': SEED,
    'n_real_requested': N_REAL, 'n_syn_requested': N_SYN, 'syn_per_gen': SYN_PER_GEN,
    'files': ['train_clean.jsonl','license_report.json','degraded_transform.py'],
    'notes': 'So-Fake-OOD未使用。TAMPERED除外。画像バイトは次段 (build/train cell) で行IDから取得',
}
with open(f'{BASE}/prep_report.json','w') as f:
    json.dump(prep, f, ensure_ascii=False, indent=2)
print('done ->', BASE)